# Kaggle runner (thin)

Runs the **repository's own scripts** on a Kaggle GPU; no research logic lives here. Setup: `docs/PHASE_4_GPU_ARCHITECTURE.md`.

Before running: **Accelerator = a GPU (T4/P100)**, **Internet = On**, attach the private dataset made from `dist/kaggle_repo.zip`
(`python scripts/package_for_kaggle.py`). To resume a previous session, also attach that notebook version's *output* as an input.
Use **Save Version -> Save & Run All (Commit)** so `/kaggle/working` (config, logs, checkpoints, `artifacts_*.zip`) is kept.
Edit only the parameter cell.

In [ ]:
import glob, os, shutil, subprocess, sys, time

# ---------------- parameters (edit only this cell) ----------------
EXPERIMENT_CONFIG = "configs/baseline.yaml"          # A: baseline | B: disease_consistency.yaml | C: anatomy_consistency.yaml (B/C refuse to run until Phases 6/7)
PRISM_CKPT = "/kaggle/temp/prism/model.safetensors"  # re-downloadable, deliberately NOT in /kaggle/working
EXTRA_SETS = ["model.device=cuda", f"model.finetuned_checkpoint_path={PRISM_CKPT}"]   # dotted overrides, one per --set
TASK = None                                          # None -> `task:` from the config (counterfactual | classifier_train)
MIN_VRAM_GB = 8
RUN_SMOKE_TEST = True                                # tiny-model GPU smoke test (minimal downloads)
RUN_REAL_SMOKE = False                               # + one minimal inference with the real PRISM checkpoint (needs FETCH_PRISM_CHECKPOINT or an attached copy)
FETCH_PRISM_CHECKPOINT = False                       # 3.4 GB download, size + SHA-256 verified
RUN_EXPERIMENT = False                               # the default config's input image is not packaged; set inputs via EXTRA_SETS first
RESTORE_FROM = sorted(glob.glob("/kaggle/input/*/experiments"))   # previous version's output, if attached
# ------------------------------------------------------------------

WORK, REPO = "/kaggle/working", "/kaggle/working/repo"
OUT = f"{WORK}/experiments"
os.makedirs(OUT, exist_ok=True)
os.environ.update(HF_HOME="/kaggle/temp/hf", PYTHONUNBUFFERED="1")   # model caches stay out of persisted output
FAILED = []

LOG = open(f"{OUT}/notebook_log.txt", "a")   # persisted copy of everything printed below

def say(text=""):
    print(text, flush=True)
    LOG.write(text + "\n")
    LOG.flush()

def run(name, cmd):
    # stream through Python so output reaches the cell log (and the log file) however the kernel handles subprocess fds
    say(); say(f"=== {name}"); say("$ " + " ".join(cmd))
    proc = subprocess.Popen(cmd, cwd=REPO if os.path.isdir(REPO) else None, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        say(line.rstrip("\n"))
    rc = proc.wait()
    say(f"--- exit code {rc}")
    if rc:
        FAILED.append((name, rc))
    return rc

In [ ]:
hits = glob.glob("/kaggle/input/*/GIT_INFO.json")
if len(hits) != 1:
    say(f"/kaggle/input contains: {sorted(os.listdir('/kaggle/input')) if os.path.isdir('/kaggle/input') else 'MISSING'}")
    raise AssertionError(f"attach exactly one repo dataset (built by scripts/package_for_kaggle.py); found {hits}")
SRC = os.path.dirname(hits[0])
run("bootstrap (copy repo, install deps, torch guarded)", [sys.executable, f"{SRC}/scripts/kaggle_bootstrap.py", "--repo-src", SRC, "--workdir", REPO])

In [ ]:
run("environment check", [sys.executable, "scripts/check_environment.py", "--require-gpu", "--min-vram-gb", str(MIN_VRAM_GB), "--out", f"{OUT}/env_report.json"])

In [ ]:
if FETCH_PRISM_CHECKPOINT:
    run("fetch PRISM checkpoint", [sys.executable, "scripts/download_prism_checkpoint.py", "--dest", os.path.dirname(PRISM_CKPT)])

In [ ]:
if RUN_SMOKE_TEST:
    cmd = [sys.executable, "scripts/gpu_smoke_test.py", "--run-root", OUT]
    if RUN_REAL_SMOKE:
        cmd += ["--real", "--prism-checkpoint", PRISM_CKPT]
    run("GPU smoke test", cmd)

In [ ]:
if RUN_EXPERIMENT:
    cmd = [sys.executable, "scripts/run_experiment.py", "--config", EXPERIMENT_CONFIG, "--run-root", OUT]
    for item in EXTRA_SETS:
        cmd += ["--set", item]
    if TASK:
        cmd += ["--task", TASK]
    if RESTORE_FROM:
        cmd += ["--restore-from", *RESTORE_FROM]
    run("experiment", cmd)

In [ ]:
# Always archive, even if a step failed, so nothing is lost; fail loudly only afterwards.
archive = shutil.make_archive(f"{WORK}/artifacts_{time.strftime('%Y%m%d-%H%M%S')}", "zip", OUT)
say(f"artifacts: {archive} {os.path.getsize(archive) / 1e6:.2f} MB")
say(f"failed steps: {FAILED or 'none'}")
if FAILED:
    raise RuntimeError(f"failed steps: {FAILED}")